# 03 - ML Modeling

## Objective
Build a first regression model to predict `total_order_value` on delivered e-commerce orders.


In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


In [2]:
DB_PATH = Path("../data/processed/ecommerce.db")

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM delivered_sales", conn)
conn.close()

df.shape


(96478, 30)

In [3]:
df.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,payment_installments_max,customer_unique_id,customer_city,customer_state,review_score_mean,review_count,delivery_delay_days,estimated_vs_actual_diff_days,total_order_value,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2017,10,...,1.0,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,1.0,8.0,-8.0,38.71,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2018,7,...,1.0,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,1.0,13.0,-6.0,141.46,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2018,8,...,3.0,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,1.0,9.0,-18.0,179.12,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2017,11,...,1.0,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,1.0,13.0,-13.0,72.20,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2018,2,...,1.0,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,1.0,2.0,-10.0,28.62,0


In [4]:
df.columns.tolist()


['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'purchase_year',
 'purchase_month',
 'purchase_day',
 'purchase_hour',
 'purchase_dayofweek',
 'n_items',
 'n_unique_products',
 'n_unique_sellers',
 'total_price',
 'total_freight',
 'n_payments',
 'payment_value_total',
 'payment_installments_max',
 'customer_unique_id',
 'customer_city',
 'customer_state',
 'review_score_mean',
 'review_count',
 'delivery_delay_days',
 'estimated_vs_actual_diff_days',
 'total_order_value',
 'is_late']

In [5]:
target = "total_order_value"


In [6]:
selected_features = [
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_hour",
    "purchase_dayofweek",
    "customer_state",
    "n_items",
    "n_unique_products",
    "n_unique_sellers",
    "payment_installments_max",
]


In [7]:
ml_df = df[selected_features + [target]].copy()
ml_df.isna().sum()


purchase_year               0
purchase_month              0
purchase_day                0
purchase_hour               0
purchase_dayofweek          0
customer_state              0
n_items                     0
n_unique_products           0
n_unique_sellers            0
payment_installments_max    1
total_order_value           0
dtype: int64

In [8]:
ml_df = ml_df.dropna().copy()
ml_df.shape


(96477, 11)

In [9]:
X = ml_df[selected_features]
y = ml_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (96477, 10)
y shape: (96477,)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (77181, 10)
X_test: (19296, 10)


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["customer_state"]
numeric_features = [
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_hour",
    "purchase_dayofweek",
    "n_items",
    "n_unique_products",
    "n_unique_sellers",
    "payment_installments_max",
]


In [12]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


In [13]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)


In [14]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [15]:
y_pred = pipeline.predict(X_test)


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.4f}")


MAE: 98.10
RMSE: 196.61
R2: 0.1458


In [17]:
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
})

results.head(10)


,actual,predicted
0,80.33,115.886501
1,45.61,130.189820
2,154.73,116.416737
3,64.11,103.796245
4,289.30,764.091587
5,44.00,103.102964
6,111.91,355.303280
7,49.10,116.576037
8,84.93,103.483245
9,157.60,130.887792


In [18]:
results["error"] = results["actual"] - results["predicted"]
results["abs_error"] = results["error"].abs()

results.sort_values("abs_error", ascending=False).head(10)


,actual,predicted,error,abs_error
12618,4681.78,362.674239,4319.105761,4319.105761
6963,4194.76,340.124486,3854.635514,3854.635514
2464,4016.91,306.458349,3710.451651,3710.451651
12850,3155.82,114.935816,3040.884184,3040.884184
10798,3358.24,367.192104,2991.047896,2991.047896
12845,3351.35,426.440820,2924.909180,2924.909180
9365,3297.40,374.736181,2922.663819,2922.663819
14653,2938.17,102.465717,2835.704283,2835.704283
10215,3126.50,363.949086,2762.550914,2762.550914
16288,2787.87,88.606458,2699.263542,2699.263542


In [19]:
encoded_cat_features = pipeline.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

all_feature_names = numeric_features + list(encoded_cat_features)

importances = pipeline.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

feature_importance_df.head(15)


,feature,importance
8,payment_installments_max,0.342414
2,purchase_day,0.149621
5,n_items,0.127506
4,purchase_dayofweek,0.090248
1,purchase_month,0.068048
3,purchase_hour,0.067290
27,customer_state_RJ,0.025897
0,purchase_year,0.020305
34,customer_state_SP,0.010490
19,customer_state_MG,0.009114


## Conclusion de l'étape 7

Cette étape a permis de construire une première baseline de régression pour prédire la valeur totale d’une commande à partir d’un ensemble limité de variables temporelles, géographiques et structurelles. Le pipeline mis en place avec `scikit-learn` permet de gérer proprement le prétraitement des données, l’encodage des variables catégorielles et l’entraînement du modèle dans une logique reproductible.

Les résultats montrent que le modèle capte une partie du signal, mais reste encore limité dans sa capacité à expliquer précisément la valeur des commandes, en particulier pour les montants très élevés. Cette baseline reste néanmoins importante, car elle valide la démarche de modélisation sans fuite de données et met en évidence plusieurs pistes d’amélioration, notamment l’enrichissement des variables explicatives et l’expérimentation de modèles plus performants.
